In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd

In [2]:
# Load Dataset
df = sns.load_dataset("tips")

df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [4]:
df.isnull().sum()

total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64

In [5]:
df.describe()

,total_bill,tip,size
count,244.000000,244.000000,244.000000
mean,19.785943,2.998279,2.569672
std,8.902412,1.383638,0.951100
min,3.070000,1.000000,1.000000
25%,13.347500,2.000000,2.000000
50%,17.795000,2.900000,2.000000
75%,24.127500,3.562500,3.000000
max,50.810000,10.000000,6.000000


In [6]:
X = df.drop("time", axis=1)
y = df["time"]

In [7]:
X

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4
...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,3
240,27.18,2.00,Female,Yes,Sat,2
241,22.67,2.00,Male,Yes,Sat,2
242,17.82,1.75,Male,No,Sat,2


In [8]:
y

0      Dinner
1      Dinner
2      Dinner
3      Dinner
4      Dinner
        ...  
239    Dinner
240    Dinner
241    Dinner
242    Dinner
243    Dinner
Name: time, Length: 244, dtype: category
Categories (2, object): ['Lunch', 'Dinner']

In [9]:
## One-Hot Encode the Categorical Features
X = pd.get_dummies(
    X,
    columns=["sex", "smoker", "day"],
    drop_first=True,
    dtype=int
)

In [10]:
X.head()

,total_bill,tip,size,sex_Female,smoker_No,day_Fri,day_Sat,day_Sun
0,16.99,1.01,2,1,1,0,0,1
1,10.34,1.66,3,0,1,0,0,1
2,21.01,3.50,3,0,1,0,0,1
3,23.68,3.31,2,0,1,0,0,1
4,24.59,3.61,4,1,1,0,0,1


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.10,random_state=42)

In [13]:
X_train

,total_bill,tip,size,sex_Female,smoker_No,day_Fri,day_Sat,day_Sun
120,11.69,2.31,2,0,1,0,0,0
10,10.27,1.71,2,0,1,0,0,1
73,25.28,5.00,2,1,0,0,1,0
159,16.49,2.00,4,0,1,0,0,1
156,48.17,5.00,6,0,1,0,0,1
...,...,...,...,...,...,...,...,...
106,20.49,4.06,2,0,0,0,1,0
14,14.83,3.02,2,1,1,0,0,1
92,5.75,1.00,2,1,0,1,0,0
179,34.63,3.55,2,0,0,0,0,1


In [14]:
from sklearn.naive_bayes import GaussianNB

In [15]:
gnb=GaussianNB()

In [16]:
gnb.fit(X_train,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [17]:
y_pred=gnb.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [19]:
print(confusion_matrix(y_pred,y_test))
print(accuracy_score(y_pred,y_test))
print(classification_report(y_pred,y_test))

[[16  0]
 [ 1  8]]
0.96
              precision    recall  f1-score   support

      Dinner       0.94      1.00      0.97        16
       Lunch       1.00      0.89      0.94         9

    accuracy                           0.96        25
   macro avg       0.97      0.94      0.96        25
weighted avg       0.96      0.96      0.96        25



In [20]:
from sklearn.model_selection import GridSearchCV

In [21]:
# Hyperparameter Grid
param_grid = {
    "var_smoothing": [1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6]
}

# param_grid = {
#     "var_smoothing": np.logspace(0, -12, 100)
# }

In [22]:
# Grid Search
grid = GridSearchCV(
    estimator=gnb,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [23]:
grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GaussianNB()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'var_smoothing': [1e-12, 1e-11, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate paramete

In [24]:
print("Best Parameter :", grid.best_params_)
print("Best CV Score :", grid.best_score_)

Best Parameter : {'var_smoothing': 1e-12}
Best CV Score : 0.9451374207188161


In [25]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

In [26]:
print(confusion_matrix(y_pred,y_test))
print(accuracy_score(y_pred,y_test))
print(classification_report(y_pred,y_test))

[[16  0]
 [ 1  8]]
0.96
              precision    recall  f1-score   support

      Dinner       0.94      1.00      0.97        16
       Lunch       1.00      0.89      0.94         9

    accuracy                           0.96        25
   macro avg       0.97      0.94      0.96        25
weighted avg       0.96      0.96      0.96        25



In [27]:
## check overfitting and underfitting

# Training Prediction
y_train_pred = gnb.predict(X_train)

# Testing Prediction
y_test_pred = gnb.predict(X_test)

# Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Training Accuracy :", train_accuracy)
print("Testing Accuracy  :", test_accuracy)

Training Accuracy : 0.9452054794520548
Testing Accuracy  : 0.96


In [28]:
difference = train_accuracy - test_accuracy

print("Difference :", difference)

if train_accuracy > 0.95 and difference > 0.10:
    print("Model is Overfitting")
elif train_accuracy < 0.80 and test_accuracy < 0.80:
    print("Model is Underfitting")
else:
    print("Model is Well Fitted")

Difference : -0.014794520547945167
Model is Well Fitted


In [29]:
difference = abs(train_accuracy - test_accuracy)

print("Training Accuracy :", train_accuracy)
print("Testing Accuracy  :", test_accuracy)
print("Difference :", difference)

if difference <= 0.05:
    print("Model is Well Fitted")
elif train_accuracy > test_accuracy:
    print("Model is Overfitting")
else:
    print("Model may be Slightly Underfitting or Test Split is Easier")

Training Accuracy : 0.9452054794520548
Testing Accuracy  : 0.96
Difference : 0.014794520547945167
Model is Well Fitted


In [30]:
difference = abs(train_accuracy - test_accuracy)

print("Training Accuracy :", train_accuracy)
print("Testing Accuracy  :", test_accuracy)
print("Difference :", difference)

if difference <= 0.05:
    print("Model is Well Fitted")
elif train_accuracy > test_accuracy:
    print("Model is Overfitting")
else:
    print("Model may be Slightly Underfitting or Test Split is Easier")

Training Accuracy : 0.9452054794520548
Testing Accuracy  : 0.96
Difference : 0.014794520547945167
Model is Well Fitted


In [31]:
import pandas as pd

while True:

    # User Input
    total_bill = float(input("Enter Total Bill: "))
    tip = float(input("Enter Tip: "))
    sex = input("Enter Sex (Male/Female): ").title()
    smoker = input("Enter Smoker (Yes/No): ").title()
    day = input("Enter Day (Thur/Fri/Sat/Sun): ").title()
    size = int(input("Enter Size: "))

    # Create DataFrame
    new_data = pd.DataFrame({
        "total_bill": [total_bill],
        "tip": [tip],
        "sex": [sex],
        "smoker": [smoker],
        "day": [day],
        "size": [size]
    })

    # One-Hot Encoding
    new_data = pd.get_dummies(
        new_data,
        columns=["sex", "smoker", "day"],
        drop_first=True,
        dtype=int
    )

    # Match columns with training data
    new_data = new_data.reindex(columns=X.columns, fill_value=0)

    # Prediction
    prediction = best_model.predict(new_data)
    probability = best_model.predict_proba(new_data)

    print("\n==============================")
    print("Predicted Time :", prediction[0])
    print("Prediction Probability :", probability.max() * 100, "%")
    print("==============================")

    # Continue or Exit
    choice = input("\nDo you want to predict again? (yes/no): ").lower()

    if choice != "yes":
        print("\nThank You!")
        break


Predicted Time : Lunch
Prediction Probability : 99.99999994323012 %

Thank You!


In [33]:
import pickle

pickle.dump(gnb, open('gaussian_nb.pkl', 'wb'))
pickle.dump(X.columns.tolist(), open('columns.pkl', 'wb'))